# Prompting Basics: Zero-Shot এবং Few-Shot

few-shot in-context learning-এর একটা প্রকৃত, একদম শুরু থেকে (from-scratch) demonstration — এটা কোনো simulation নয়। আমরা একটা ছোট্ট decoder-only Transformer-কে (Phase 02-এর mini-GPT-র মতোই architecture family) অসংখ্য random-generated EPISODE-এর ওপর train করি, যেখানে প্রতিটি episode একটা সাধারণ function-mapping task-এর নতুন, random-parameterized instance — "y = (x + k) mod M", একটা hidden shift k সহ। Episode-এর prompt ওই নির্দিষ্ট k-এর জন্য কয়েকটা (x -> y) example pair দেখায়, আর তারপর একটা query x যেটা মডেলকে সঠিক y-তে map করতে হবে।

যেহেতু k প্রতিটি episode-এ নতুন করে sample করা হয়, মডেল তার weights-এ কখনো fixed input->output table মুখস্থ করতে পারে না — query-র সঠিক উত্তর দেওয়ার একমাত্র উপায় হলো সেই নির্দিষ্ট prompt-এ দেওয়া in-context examples থেকে k INFER করে প্রয়োগ করা। এটাই হলো GPT-3-এর আচরণের একটা ছোট্ট, সৎ analogue, যখন সে কয়েকটা prompt example থেকে কোনো gradient update ছাড়াই একটা নতুন task সম্পন্ন করে (Phase 03 Lesson 1, section 3)।

Training-এর পরে আমরা মডেলকে test করি সেসব shift value k দিয়ে যেগুলো সে training-এ **কখনো** দেখেনি — একটা নতুন function-এর প্রতি সত্যিকারের zero-shot generalization, সম্পূর্ণরূপে inference-এ in-context examples-এর মাধ্যমে evaluate করা, মডেলের weights একদম frozen অবস্থায়। তারপর আমরা মাপি কিভাবে accuracy নির্ভর করে (a) কয়টা in-context example দেওয়া হলো, আর (b) তাদের ORDER training-এর canonical order-এর সাথে মেলে কি না — যেটা Zhao et al. (2021), "Calibrate Before Use"-এ নথিভুক্ত prompt-sensitivity ঘটনার সরাসরি, পরিমাপ-করা demonstration।

**Runtime:** CPU-তে মোটামুটি 20-40 সেকেন্ড (ছোট sequences-এ 3000 training steps)।

**চালানোর নিয়ম:**
- এই notebook-এর প্রতিটি cell আগের cell-গুলোর ওপর নির্ভর করে — উপর থেকে নিচে ক্রমান্বয়ে চালাও।
- আসল Python স্ক্রিপ্ট চালাতে চাইলে: `python example.py`

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## Section 0 — Task family: y = (x + k) mod M

`M = 6` টি symbol-এর একটা alphabet, আর task হলো `y = (x + k) mod M` — একটা hidden shift `k` সহ। `TRAIN_KS` হলো সেই shift-গুলো যেগুলোতে মডেল train করে; `TEST_KS` সম্পূর্ণভাবে held out — মডেলকে test-time-এ purely in-context inference দিয়ে সেগুলোতে generalize করতে হবে। নিচের cell-এ alphabet, encoding, episode-বিল্ডিং helper-গুলো define করা হয়েছে।

In [ ]:
# ---------------------------------------------------------------------------
# 0. Task family: y = (x + k) mod M
#
# M = symbol alphabet-এর size। k = hidden "shift", যা একটা episode-এর
# function-কে সংজ্ঞায়িত করে। TRAIN_KS হলো সেই shift-গুলো যেগুলো মডেল
# training-এ দেখে; TEST_KS সম্পূর্ণভাবে held out থাকে -- মডেলকে test-time-এ
# purely in-context inference দিয়ে সেগুলোতে generalize করতে হবে।
# ---------------------------------------------------------------------------

M = 6
TRAIN_KS = [0, 1, 2, 3]
TEST_KS = [4, 5]          # training-এ কখনো আসে না -- সত্যিকারের novel functions

CHARS = [str(d) for d in range(M)] + [">", ","]
vocab_size = len(CHARS)
stoi = {ch: i for i, ch in enumerate(CHARS)}
itos = {i: ch for i, ch in enumerate(CHARS)}


def encode(s):
    return [stoi[ch] for ch in s]


def build_episode_string(k, x_shown, x_query):
    """x_shown: few-shot examples হিসেবে দেখানোর জন্য distinct x মান-গুলোর
    list (দেওয়া ক্রমে -- caller sorted বনাম scrambled নিয়ন্ত্রণ করে)।
    সম্পূর্ণ prompt string (ঠিক 'x_query>' এর পরে শেষ) এবং সঠিক উত্তরের
    digit-টা string হিসেবে return করে।"""
    parts = []
    for x in x_shown:
        y = (x + k) % M
        parts.append(f"{x}>{y},")
    prompt = "".join(parts) + f"{x_query}>"
    answer = str((x_query + k) % M)
    return prompt, answer


def sample_episode(k, n_shown, scrambled=False):
    xs = list(range(M))
    random.shuffle(xs)
    x_shown = xs[:n_shown]
    x_query = xs[n_shown]                      # shown-থেকে distinct হওয়া নিশ্চিত
    if not scrambled:
        x_shown = sorted(x_shown)               # training-এ ব্যবহৃত canonical order
    prompt, answer = build_episode_string(k, x_shown, x_query)
    return prompt, answer

## Section 1 — Hyperparameters এবং model (Phase 02-এর decoder-only recipe)

Phase 02 Lesson 6-এর একই decoder-only Transformer recipe — causal self-attention + feed-forward blocks — এই lesson-টা self-contained রাখতে এখানে re-declare করা হয়েছে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Hyperparameters এবং model -- Phase 02 Lesson 6-এর একই decoder-only
#    recipe, এই lesson-টা self-contained রাখতে এখানে আবার declare করা হয়েছে।
# ---------------------------------------------------------------------------

BLOCK_SIZE = 24     # "4*5(examples) + 2(query) + 1(answer)" = 23 পর্যন্ত max length
D_MODEL = 64
NUM_HEADS = 4
D_FF = 4 * D_MODEL
NUM_LAYERS = 3
BATCH_SIZE = 64
NUM_ITERS = 3000
LEARNING_RATE = 3e-3


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).bool())

    def forward(self, x):
        batch, T, d_model = x.shape

        def split_heads(t):
            return t.view(batch, T, self.num_heads, self.d_k).transpose(1, 2)

        Q, K, V = split_heads(self.W_q(x)), split_heads(self.W_k(x)), split_heads(self.W_v(x))
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(batch, T, d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, block_size) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, targets=None):
        batch, T = token_ids.shape
        positions = torch.arange(T, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.output_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def predict_next_char(self, prompt_str):
        """পরের character-টার ওপর greedy argmax -- এটাই সেই in-context
        'answer' যেটা মডেল commit করে, কোনো weight update ছাড়াই।"""
        ids = torch.tensor([encode(prompt_str)], dtype=torch.long)
        logits, _ = self(ids)
        next_id = logits[0, -1, :].argmax().item()
        return itos[next_id]

## Section 2 — Batch generation: প্রতিটি step-এ fresh random episode

প্রতিটি training example-এর জন্য সম্পূর্ণ নতুন random `(k, examples)` episode তৈরি হয় — এখানে কিছুই verbatim-এ repeat হয় না, যেটা মডেলকে fixed lookup table মুখস্থ করার বদলে algorithm (context থেকে k অনুমান করে প্রয়োগ) শিখতে বাধ্য করে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Batch generation -- প্রতিটি training example-এর জন্য fresh random
#    (k, examples) episode। এখানে কিছুই verbatim-এ repeat হয় না, আর এটাই
#    মডেলকে ALGORITHM শিখতে বাধ্য করে (context থেকে k অনুমান করে তারপর
#    প্রয়োগ), input->output pairs মুখস্থ করার বদলে।
# ---------------------------------------------------------------------------

def get_training_batch(batch_size):
    n_shown = random.randint(2, 5)              # প্রতি step-এ বদলায়, একটা batch-এ স্থির
    seqs = []
    for _ in range(batch_size):
        k = random.choice(TRAIN_KS)
        prompt, answer = sample_episode(k, n_shown, scrambled=False)
        seqs.append(encode(prompt + answer))
    tokens = torch.tensor(seqs, dtype=torch.long)
    return tokens[:, :-1], tokens[:, 1:]

## Section 3 — Evaluation helpers

Held-out shifts-এ accuracy মাপার helper — প্রতিটি trial-এ একটা fresh episode তৈরি করে model-এর prediction-কে সঠিক উত্তরের সাথে তুলনা করা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Evaluation helpers
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, ks, n_shown, scrambled, num_trials=300):
    correct = 0
    for _ in range(num_trials):
        k = random.choice(ks)
        prompt, answer = sample_episode(k, n_shown, scrambled=scrambled)
        pred = model.predict_next_char(prompt)
        correct += int(pred == answer)
    return correct / num_trials

## পুরো demonstration চালানো

নিচের cell-এ `main()` function define করা হয়েছে — setup তথ্য, training-এর আগে/পরে held-out shifts-এ accuracy, in-context examples-এর সংখ্যার প্রভাব, আর example order-এর প্রভাব (Zhao et al., 2021)। শেষ cell-এ `main()` কল করে সম্পূর্ণ demo run হয়।

In [ ]:
def main():
    print("=" * 70)
    print("SETUP")
    print("=" * 70)
    print(f"Alphabet size M = {M}   (symbols 0..{M-1})")
    print(f"Function family: y = (x + k) mod {M}, for hidden shift k")
    print(f"Training shifts (seen during training):     k in {TRAIN_KS}")
    print(f"Held-out shifts (NEVER seen during training): k in {TEST_KS}")
    print(f"Vocabulary ({vocab_size} characters): {CHARS}")
    example_prompt, example_answer = sample_episode(2, 3, scrambled=False)
    print(f"\nExample training-style prompt (k=2, 3 shown examples):")
    print(f"  {example_prompt!r}  -> correct next char: {example_answer!r}")

    model = MiniGPT(vocab_size, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameter count: {num_params:,}")

    print("\n" + "=" * 70)
    print("BEFORE TRAINING: accuracy on held-out shifts (random weights)")
    print("=" * 70)
    random_baseline = 1.0 / M
    acc_before = evaluate(model, TEST_KS, n_shown=4, scrambled=False, num_trials=300)
    print(f"Chance accuracy (uniform guess over {M} symbols): {random_baseline:.3f}")
    print(f"Untrained model accuracy:                          {acc_before:.3f}")

    print("\n" + "=" * 70)
    print("TRAINING (next-token prediction over randomly generated episodes)")
    print("=" * 70)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    for step in range(1, NUM_ITERS + 1):
        x, y = get_training_batch(BATCH_SIZE)
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 300 == 0 or step == 1:
            print(f"  step {step:5d}  loss = {loss.item():.4f}")

    print("\n" + "=" * 70)
    print("AFTER TRAINING: in-context generalization to shifts NEVER seen")
    print("during training (weights are now completely frozen)")
    print("=" * 70)
    acc_after = evaluate(model, TEST_KS, n_shown=4, scrambled=False, num_trials=300)
    print(f"Held-out-shift accuracy with 4 in-context examples: {acc_after:.3f}")
    print(f"(chance level is {random_baseline:.3f}; {M-1}/{M} of guesses would be wrong at random)")

    print("\nA few concrete examples on shift k=5 (held out, never trained on):")
    for _ in range(4):
        prompt, answer = sample_episode(5, n_shown=4, scrambled=False)
        pred = model.predict_next_char(prompt)
        mark = "correct" if pred == answer else "WRONG"
        print(f"  prompt={prompt!r:28s} true={answer}  model={pred}  [{mark}]")

    if acc_after > acc_before + 0.1:
        print("\n-> The model was NEVER trained on k=4 or k=5. Every correct answer")
        print("   above comes purely from reading the in-context examples in THAT")
        print("   prompt and inferring k on the fly -- exactly the mechanism behind")
        print("   GPT-3's few-shot in-context learning (Phase 03 Lesson 1 section 3),")
        print("   just demonstrated end-to-end on a task small enough to train here.")

    print("\n" + "=" * 70)
    print("NUMBER OF IN-CONTEXT EXAMPLES vs. ACCURACY (held-out shifts)")
    print("=" * 70)
    print("Training only ever showed between 2 and 5 examples per episode.")
    print(f"{'n_shown':>10}{'accuracy':>12}")
    counts_results = {}
    for n_shown in [1, 2, 3, 4, 5]:
        acc = evaluate(model, TEST_KS, n_shown=n_shown, scrambled=False, num_trials=300)
        counts_results[n_shown] = acc
        note = "  (below training range)" if n_shown < 2 else ""
        print(f"{n_shown:>10}{acc:>12.3f}{note}")

    print("\n-> Accuracy rises as more in-context examples are given, and is at its")
    print("   worst with only 1 example -- a prompt shorter than anything the model")
    print("   ever trained on. More examples give the model more redundant evidence")
    print("   to pin down the hidden shift k before it has to commit to an answer.")

    print("\n" + "=" * 70)
    print("EXAMPLE ORDER vs. ACCURACY: sorted (training distribution) vs.")
    print("scrambled (never seen during training) -- Zhao et al. (2021)")
    print("=" * 70)
    N_SHOWN_FOR_ORDER_TEST = 4
    acc_sorted = evaluate(model, TEST_KS, n_shown=N_SHOWN_FOR_ORDER_TEST, scrambled=False, num_trials=500)
    acc_scrambled = evaluate(model, TEST_KS, n_shown=N_SHOWN_FOR_ORDER_TEST, scrambled=True, num_trials=500)
    print(f"Sorted order   (matches training):     accuracy = {acc_sorted:.3f}")
    print(f"Scrambled order (never seen in training): accuracy = {acc_scrambled:.3f}")

    if acc_scrambled < acc_sorted - 0.03:
        print("\n-> Same examples, same hidden function, same COUNT of examples -- the")
        print("   only thing that changed is the ORDER they appear in the prompt, and")
        print("   accuracy measurably drops. The model was only ever trained on")
        print("   examples sorted by x, so it partly relies on that positional")
        print("   regularity rather than a perfectly order-invariant algorithm. This")
        print("   is a small, controlled reproduction of the prompt-order sensitivity")
        print("   Zhao et al. (2021, 'Calibrate Before Use') documented in real LLMs:")
        print("   logically equivalent prompts that differ only in example order or")
        print("   formatting can produce measurably different accuracy.")
    else:
        print("\n-> Here, scrambling order did NOT meaningfully hurt accuracy: the model")
        print("   generalized the underlying algorithm (infer k from ANY example pair,")
        print("   then apply it) well enough that it doesn't depend on position. Real")
        print("   LLMs are not always this robust -- Zhao et al. (2021) found that with")
        print("   less redundant, more ambiguous tasks than this one, example order and")
        print("   formatting DO measurably move accuracy, which is why prompt order is")
        print("   worth controlling for rather than assuming it is harmless.")

In [ ]:
main()